# INFORMATION SECURITY COURSE PROJECT REPORT
## RESEARCH AND IMPLEMENTATION OF MODERN CRYPTOGRAPHIC PRIMITIVES FROM SCRATCH

* **Student Name:** Nguyen Thi Nhu Y
* **Student ID (SID):** Ends with `0`
* **Runtime Environment:** Python 3.13 (CPython 64-bit, Windows 11)
* **Academic Integrity Rule:** 100% Python Standard Library only (`secrets`, `math`, `time`, `typing`). Absolutely NO external cryptographic libraries (`cryptography`, `pycryptodome`, `hashlib`, `rsa`, etc.).
* **Standardized Cryptographic Execution Order:**
  1. **AES-128** (Symmetric Block Cipher - NIST FIPS 197, CBC mode, PKCS#7 padding)
  2. **RSA** (Asymmetric Public-Key Cryptosystem - RFC 8017 / PKCS #1 v2.2)
  3. **SHA-256** (Cryptographic Hash Function - NIST FIPS 180-4)
  4. **ElGamal** (Digital Signature Scheme - Taher Elgamal 1985)

---

---
# SECTION 0: ACADEMIC INTEGRITY AUDIT AND STATIC AST SCAN

The project strictly prohibits third-party cryptographic libraries (`cryptography`, `pycryptodome`, `hashlib`, `rsa`, `ssl`, etc.) as well as dynamic import primitives (`__import__`, `eval`, `exec`).
All 15 Python files in the repository are statically verified using Python's `ast` (Abstract Syntax Tree) module via `tools/check_forbidden_imports.py`.


In [2]:
# Environment setup and AST static security scan
import os
import sys
sys.path.insert(0, os.path.abspath('.'))

from tools.check_forbidden_imports import scan_directory
total_scanned, violations = scan_directory(os.path.abspath('.'))
if not violations:
    print(f"[PASS] Academic Integrity Verified: 0 forbidden imports detected across all {total_scanned} Python source files.")
else:
    print(f"[FAIL] Detected {len(violations)} violations!")


[PASS] Academic Integrity Verified: 0 forbidden imports detected across all 15 Python source files.


---
# SECTION 1: AES-128 SYMMETRIC BLOCK CIPHER (NIST FIPS 197)

## 1.1. Historical Context, Development, and Design Rationale
The Advanced Encryption Standard (AES) was officially published by the National Institute of Standards and Technology (NIST) in 2001 in FIPS PUB 197. Selected through a multi-year worldwide cryptographic competition, the Rijndael algorithm (developed by Belgian cryptographers Joan Daemen and Vincent Rijmen) replaced the aging Data Encryption Standard (DES). AES features a fixed 128-bit block size and supports 128, 192, and 256-bit key lengths. AES-128 uses 10 rounds of transformation, balancing high computational efficiency with robust resistance against differential and linear cryptanalysis.

## 1.2. Mathematical Foundations on Galois Field GF(2^8)
AES operates on a $4 \times 4$ byte state matrix where each byte represents an element in the finite Galois Field $\text{GF}(2^8) \cong \mathbb{F}_2[x] / (m(x))$ with the irreducible Rijndael polynomial:
$$m(x) = x^8 + x^4 + x^3 + x + 1 \quad (\text{hexadecimal } \mathtt{0x11B})$$
- **Field Addition/Subtraction:** Equivalent to bitwise XOR $(\oplus)$.
- **Field Multiplication:** Achieved through polynomial multiplication modulo $m(x)$ using xtime operations:
$$\mathtt{xtime}(a) = (a \ll 1) \oplus (\mathtt{0x1B} \text{ if } (a \& \mathtt{0x80}) \text{ else } 0)$$

## 1.3. Round Transformations (SubBytes, ShiftRows, MixColumns, AddRoundKey)
Each intermediate round (1 through 9) executes four invertible mathematical operations:
1. **SubBytes:** Non-linear byte substitution using a 256-byte S-Box. The S-Box computes the multiplicative inverse in $\text{GF}(2^8)$ followed by an affine transformation over $\text{GF}(2)$, providing optimal non-linearity against linear and differential attacks.
2. **ShiftRows:** Cyclic left shift of rows: Row 0 shifted by 0, Row 1 by 1, Row 2 by 2, and Row 3 by 3 bytes, ensuring inter-column diffusion.
3. **MixColumns:** Column-wise matrix multiplication over $\text{GF}(2^8)$ using the fixed circulant MDS matrix:
$$\begin{bmatrix} s'_{0,c} \\ s'_{1,c} \\ s'_{2,c} \\ s'_{3,c} \end{bmatrix} = \begin{bmatrix} 02 & 03 & 01 & 01 \\ 01 & 02 & 03 & 01 \\ 01 & 01 & 02 & 03 \\ 03 & 01 & 01 & 02 \end{bmatrix} \begin{bmatrix} s_{0,c} \\ s_{1,c} \\ s_{2,c} \\ s_{3,c} \end{bmatrix}$$
4. **AddRoundKey:** Bitwise XOR of the state matrix with the 128-bit round subkey derived from Key Expansion.

## 1.4. Key Expansion Algorithm
Expands the 16-byte initial cipher key into 11 round keys (44 32-bit words $w_0, \dots, w_{43}$) using `RotWord`, `SubWord`, and round constants `Rcon`:
$$w_i = w_{i-1} \oplus w_{i-4} \quad (i \not\equiv 0 \pmod 4)$$
$$w_i = \mathtt{SubWord}(\mathtt{RotWord}(w_{i-1})) \oplus \mathtt{Rcon}[i/4] \oplus w_{i-4} \quad (i \equiv 0 \pmod 4)$$

## 1.5. Cipher Block Chaining (CBC) Mode & PKCS#7 Padding
For variable-length plaintexts, NIST SP 800-38A defines the CBC mode with an unpredictable 16-byte Initialization Vector (IV):
$$C_0 = \text{IV}, \quad C_i = E_K(P_i \oplus C_{i-1}), \quad P_i = D_K(C_i) \oplus C_{i-1}$$
PKCS#7 pads the message to a multiple of 16 bytes by appending $b$ bytes, each having value $b$ ($1 \le b \le 16$). A full 16-byte block is added if the message is already an exact multiple of 16 bytes.

## 1.6. Security Analysis & Vulnerabilities
- **Malleability in CBC:** Modifying byte $j$ in ciphertext block $C_{i-1}$ introduces deterministic bit flips in plaintext $P_i$, while completely scrambling $P_{i-1}$.
- **Padding Oracle Attacks:** If an attacker can distinguish valid from invalid PKCS#7 padding responses, they can decrypt arbitrary ciphertexts without the key.


In [4]:
# AES-128 Verification: Key Expansion, FIPS 197 C.1, CBC + PKCS#7, and Tamper Detection
from src.aes128 import key_expansion, encrypt_block, decrypt_block, aes128_encrypt_message, aes128_decrypt_message
from src.sha256 import sha256_bytes

# 1. Student key derivation
student_name = "Nguyen Thi Nhu Y"
key = sha256_bytes(student_name.encode('utf-8'))[:16]
round_keys = key_expansion(key)
print(f"[+] Student AES-128 Key (16B): {key.hex()}")
print(f"[+] Key Expansion: Successfully generated {len(round_keys)} round subkeys.")

# 2. FIPS 197 Appendix C.1 Single Block Verification
fips_k = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
fips_p = bytes.fromhex("00112233445566778899aabbccddeeff")
fips_expected = bytes.fromhex("69c4e0d86a7b0430d8cdb78070b4c55a")
fips_rkeys = key_expansion(fips_k)
fips_c = encrypt_block(fips_p, fips_rkeys)
print(f"[+] NIST FIPS 197 App C.1 Match: {fips_c == fips_expected} (Ciphertext: {fips_c.hex()})")

# 3. CBC Mode + PKCS#7 Encryption & Decryption
msg = b"Information Security 2026 - Cryptography From Scratch - Nguyen Thi Nhu Y"
cipher = aes128_encrypt_message(msg, key)
decrypted = aes128_decrypt_message(cipher, key)
print(f"[+] CBC + PKCS#7 Roundtrip Success: {decrypted == msg}")

# 4. Tamper Detection
tampered = bytearray(cipher)
tampered[-1] ^= 0x01  # Corrupt padding byte
try:
    aes128_decrypt_message(bytes(tampered), key)
    print("[-] Tamper test FAILED (No padding error)")
except ValueError as e:
    print(f"[+] Tamper Detection Verified: Caught invalid PKCS#7 padding exception ({e})")


[+] Student AES-128 Key (16B): f5db6c7e3f6d978c01b2409cd300d3c5
[+] Key Expansion: Successfully generated 11 round subkeys.
[+] NIST FIPS 197 App C.1 Match: True (Ciphertext: 69c4e0d86a7b0430d8cdb78070b4c55a)
[+] CBC + PKCS#7 Roundtrip Success: True
[+] Tamper Detection Verified: Caught invalid PKCS#7 padding exception (Invalid PKCS#7 padding byte value)


## 1.8. Empirical Evaluation and Experimental Test Results: AES-128
- **Automated Unit Testing:** 33 out of 33 tests in `tests/test_aes128.py` PASSED completely.
- **Benchmark Performance:**
  - `key_expansion`: 37,207.9 ops/s (Median 0.0263 ms, $n=50$).
  - CBC Encryption throughput: 0.274 - 0.279 MB/s across 1 KB, 16 KB, and 64 KB.
  - CBC Decryption throughput: 0.026 - 0.027 MB/s (slower due to larger polynomial coefficients in `InvMixColumns`).

### Empirical Terminal Screenshot: AES-128 Benchmark Execution
![AES-128 Benchmark Terminal](evidence/benchmark_aes128.png)

### Empirical Terminal Screenshot: AES-128 Interactive Demo Execution
![AES-128 Demo Terminal](evidence/demo_aes128.jpg)


---
# SECTION 2: RSA ASYMMETRIC CRYPTOSYSTEM (RFC 8017 / PKCS #1)

## 2.1. Historical Context, Development, and Design Rationale
Proposed in 1977 by Ron Rivest, Adi Shamir, and Leonard Adleman at MIT, RSA is the first practical public-key cryptosystem enabling both confidential communication and digital signatures without prior shared secrets. Its security rests on the computational intractability of the Integer Factorization Problem (IFP).

## 2.2. Number Theoretic Foundations
Let $p, q$ be two distinct large prime numbers. The RSA modulus is $n = p \cdot q$, and Euler's totient is $\phi(n) = (p-1)(q-1)$.
The public exponent $e$ is chosen such that $1 < e < \phi(n)$ and $\gcd(e, \phi(n)) = 1$ (standard choice $e = 65537 = 2^{16}+1$).
The private exponent $d$ is computed as the modular multiplicative inverse of $e$ modulo $\phi(n)$:
$$d \cdot e \equiv 1 \pmod{\phi(n)}$$
By Euler's Theorem, for any message $m \in \mathbb{Z}_n$:
$$c \equiv m^e \pmod n, \quad m \equiv c^d \equiv (m^e)^d \equiv m^{k \phi(n) + 1} \equiv m \pmod n$$

## 2.3. Miller-Rabin Probabilistic Primality Test
Given candidate odd integer $n > 2$, write $n - 1 = 2^s \cdot d$ where $d$ is odd. A base $a$ is drawn uniformly from $[2, n-2]$. If:
$$a^d \not\equiv 1 \pmod n \quad \text{and} \quad a^{2^r d} \not\equiv -1 \pmod n \quad \forall r \in [0, s-1]$$
then $n$ is definitely composite. Running $k = 40$ independent rounds ensures the probability of a composite passing as prime is bounded by $4^{-40} \approx 8.27 \times 10^{-25}$.

## 2.4. Extended Euclidean Algorithm & Modular Multiplicative Inverse
Given integers $a, b$, the Extended Euclidean algorithm finds integers $x, y$ satisfying Bézout's identity: $a \cdot x + b \cdot y = \gcd(a, b)$.
When $\gcd(a, m) = 1$, the modular inverse is $a^{-1} \equiv x \pmod m$.

## 2.5. Security Analysis & Textbook RSA Vulnerabilities
- **Malleability of Textbook RSA:** $E(m_1) \cdot E(m_2) = (m_1^e)(m_2^e) = (m_1 m_2)^e = E(m_1 m_2) \pmod n$, allowing ciphertext tampering.
- **Small Plaintexts:** Without padding, small plaintexts $m < n^{1/e}$ can be decrypted directly via integer root extraction $\sqrt[e]{c}$. Production systems must deploy OAEP padding (PKCS #1 v2.2).


In [7]:
# RSA Verification: Miller-Rabin Primality, Keypair Generation, Integer and Bytes Roundtrips
from src.rsa import generate_rsa_keypair, rsa_encrypt, rsa_decrypt, rsa_encrypt_bytes, rsa_decrypt_bytes, is_probable_prime

# 1. Miller-Rabin Primality Verification
print(f"[+] Miller-Rabin is_probable_prime(65537, 40): {is_probable_prime(65537, 40)} (Expected True)")
print(f"[+] Miller-Rabin Carmichael is_probable_prime(561, 40): {is_probable_prime(561, 40)} (Expected False)")

# 2. Keypair generation (512-bit)
pub_key, priv_key = generate_rsa_keypair(bits=512)
e, n = pub_key
d, _ = priv_key
print(f"[+] Generated 512-bit RSA Keypair: Modulus n length = {n.bit_length()} bits")

# 3. Integer Encryption & Decryption
m_int = 202611230104062006
c_int = rsa_encrypt(m_int, pub_key)
dec_int = rsa_decrypt(c_int, priv_key)
print(f"[+] Integer Roundtrip Success: {dec_int == m_int}")

# 4. Byte string encryption & Wrong Key Detection
msg_bytes = b"Nguyen Thi Nhu Y - RSA PKCS#1 Demo"
c_bytes = rsa_encrypt_bytes(msg_bytes, pub_key)
dec_bytes = rsa_decrypt_bytes(c_bytes, priv_key)
print(f"[+] Byte String Roundtrip Success: {dec_bytes == msg_bytes}")

wrong_priv = (d + 2, n)
wrong_dec = rsa_decrypt(c_int, wrong_priv)
print(f"[+] Wrong Key Rejection Verified: Decryption mismatch = {wrong_dec != m_int}")


[+] Miller-Rabin is_probable_prime(65537, 40): True (Expected True)
[+] Miller-Rabin Carmichael is_probable_prime(561, 40): False (Expected False)
[+] Generated 512-bit RSA Keypair: Modulus n length = 512 bits
[+] Integer Roundtrip Success: True
[+] Byte String Roundtrip Success: True
[+] Wrong Key Rejection Verified: Decryption mismatch = True


## 2.8. Empirical Evaluation and Experimental Test Results: RSA
- **Automated Unit Testing:** 12 out of 12 tests in `tests/test_rsa.py` PASSED completely.
- **Benchmark Performance:**
  - **512-bit RSA:** Keygen: 18.23 ms (53.3 ops/s); Encrypt: 0.012 ms (81,125 ops/s); Decrypt: 0.467 ms (2,047 ops/s).
  - **1024-bit RSA:** Keygen: 166.79 ms (5.15 ops/s); Encrypt: 0.040 ms (22,526 ops/s); Decrypt: 2.804 ms (347 ops/s).
  - **2048-bit RSA:** Keygen: 1,202.88 ms (0.75 ops/s); Encrypt: 0.120 ms (8,317 ops/s); Decrypt: 18.064 ms (54.7 ops/s).
  - **Asymmetry:** Encryption with $e=65537$ is 38x to 150x faster than decryption with private exponent $d$.

### Empirical Terminal Screenshot: RSA Benchmark Execution
![RSA Benchmark Terminal](evidence/benchmark_rsa.png)

### Empirical Terminal Screenshot: RSA Interactive Demo Execution
![RSA Demo Terminal](evidence/demo_rsa.jpg)


---
# SECTION 3: SHA-256 CRYPTOGRAPHIC HASH FUNCTION (NIST FIPS 180-4)

## 3.1. Historical Context, Development, and Design Rationale
Designed by the National Security Agency (NSA) and published by NIST in 2002 in FIPS PUB 180-2 (updated to FIPS PUB 180-4 in 2015), SHA-256 belongs to the SHA-2 family. It replaces SHA-1 and MD5, offering 128-bit collision resistance and 256-bit preimage resistance against quantum and classical attacks.

## 3.2. Merkle-Damgård Construction & Davies-Meyer Compression
SHA-256 processes messages in 512-bit blocks ($M^{(1)}, \dots, M^{(N)}$) following the Merkle-Damgård structure with Davies-Meyer feedforward:
$$H^{(i)} = H^{(i-1)} + C(H^{(i-1)}, M^{(i)}) \pmod{2^{32}}$$
Initial hash values $H_0^{(0)}, \dots, H_7^{(0)}$ are derived from the fractional parts of the square roots of the first 8 prime numbers.

## 3.3. Message Padding & 64-word Message Schedule
1. **Padding:** Append a single `1` bit (`0x80`), followed by $k$ zero bits such that $(L + 1 + k) \equiv 448 \pmod{512}$, then append the 64-bit big-endian representation of original length $L$.
2. **Schedule Expansion:** Words $W_0, \dots, W_{15}$ are taken directly from the 512-bit block. Words $W_{16}, \dots, W_{63}$ are expanded recursively:
$$W_t = \sigma_1(W_{t-2}) + W_{t-7} + \sigma_0(W_{t-15}) + W_{t-16} \pmod{2^{32}}$$
$$\sigma_0(x) = \text{ROTR}^7(x) \oplus \text{ROTR}^{18}(x) \oplus \text{SHR}^3(x)$$
$$\sigma_1(x) = \text{ROTR}^{17}(x) \oplus \text{ROTR}^{19}(x) \oplus \text{SHR}^{10}(x)$$

## 3.4. 64-step Compression Loop & Bitwise Non-linear Functions
The eight 32-bit working variables $(a, b, c, d, e, f, g, h)$ are updated across 64 rounds using constants $K_0, \dots, K_{63}$:
$$T_1 = h + \Sigma_1(e) + \text{Ch}(e, f, g) + K_t + W_t, \quad T_2 = \Sigma_0(a) + \text{Maj}(a, b, c)$$
$$h = g, \; g = f, \; f = e, \; e = d + T_1, \; d = c, \; c = b, \; b = a, \; a = T_1 + T_2$$
$$\text{Ch}(x, y, z) = (x \land y) \oplus (\neg x \land z), \quad \text{Maj}(x, y, z) = (x \land y) \oplus (x \land z) \oplus (y \land z)$$
$$\Sigma_0(x) = \text{ROTR}^2(x) \oplus \text{ROTR}^{13}(x) \oplus \text{ROTR}^{22}(x), \quad \Sigma_1(x) = \text{ROTR}^6(x) \oplus \text{ROTR}^{11}(x) \oplus \text{ROTR}^{25}(x)$$

## 3.5. Avalanche Effect and Strict Avalanche Criterion (SAC)
A cryptographic hash function must exhibit the Avalanche Effect: flipping a single bit in the input message should cause each output bit to flip with probability $p = 0.5$ (optimal range $40\% - 60\%$).


In [10]:
# SHA-256 Verification: Standard NIST Test Vectors and Avalanche Effect Verification
from src.sha256 import sha256_hex, sha256_bytes

# 1. Standard Test Vectors
v_empty = sha256_hex(b"")
v_abc = sha256_hex(b"abc")
v_name = sha256_hex("Nguyen Thi Nhu Y")
print(f"[+] SHA-256(''):    {v_empty}")
print(f"    Matches NIST:   {v_empty == 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}")
print(f"[+] SHA-256('abc'): {v_abc}")
print(f"    Matches NIST:   {v_abc == 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'}")
print(f"[+] SHA-256('Nguyen Thi Nhu Y'): {v_name}")

# 2. Strict Avalanche Criterion (SAC) Verification
m1 = b"Information Security 2026 - Nguyen Thi Nhu Y"
m2 = bytearray(m1)
m2[-1] ^= 0x01  # Flip single bit
h1 = sha256_bytes(bytes(m1))
h2 = sha256_bytes(bytes(m2))
flipped = sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(h1, h2))
ratio = (flipped / 256.0) * 100.0
print(f"[+] Avalanche Bit Flip: {flipped}/256 ({ratio:.2f}%) - SAC Criterion: {35 <= ratio <= 65}")


[+] SHA-256(''):    e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
    Matches NIST:   True
[+] SHA-256('abc'): ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
    Matches NIST:   True
[+] SHA-256('Nguyen Thi Nhu Y'): f5db6c7e3f6d978c01b2409cd300d3c551209137b1ada810c6323122acb47d4d
[+] Avalanche Bit Flip: 134/256 (52.34%) - SAC Criterion: True


## 3.8. Empirical Evaluation and Experimental Test Results: SHA-256
- **Automated Unit Testing:** 5 out of 5 tests in `tests/test_sha256.py` PASSED completely.
- **Benchmark Performance:**
  - 1 KB: Median 2.076 ms (Throughput 0.451 MB/s, $n=20$).
  - 10 KB: Median 19.306 ms (Throughput 0.498 MB/s, $n=20$).
  - 100 KB: Median 192.579 ms (Throughput 0.505 MB/s, $n=10$).
  - 1 MB: Median 1,997.217 ms (Throughput 0.500 MB/s, $n=10$).
  - Linear asymptotic time complexity $\mathcal{O}(N)$ strictly demonstrated across all data scales.

### Empirical Terminal Screenshot: SHA-256 Benchmark Execution
![SHA-256 Benchmark Terminal](evidence/benchmark_sha256.png)

### Empirical Terminal Screenshot: SHA-256 Interactive Demo Execution
![SHA-256 Demo Terminal](evidence/demo_sha256.jpg)


---
# SECTION 4: ELGAMAL DIGITAL SIGNATURE SCHEME (TAHER ELGAMAL 1985)

## 4.1. Historical Context, Development, and Design Rationale
Published in 1985 by Taher Elgamal, this public-key signature scheme is based on the Discrete Logarithm Problem (DLP) over cyclic groups. It served as the direct theoretical predecessor for the US Digital Signature Standard (DSS/DSA) and modern Schnorr signatures.

## 4.2. Safe Prime Domain Parameters and Generator Selection
To prevent the Pohlig-Hellman small subgroup attack, $p$ is chosen as a safe prime: $p = 2q + 1$ where $q$ is also prime. In the multiplicative group $\mathbb{Z}_p^*$, the order is $p - 1 = 2q$, whose only proper divisors are $2$ and $q$.
An integer $g \in [2, p-2]$ is a primitive root generator iff:
$$g^2 \not\equiv 1 \pmod p \quad \text{and} \quad g^q \not\equiv 1 \pmod p$$

## 4.3. Keypair Generation, Signature Generation & Verification
1. **Key Generation:** Private key $x \in [2, p-2]$, public key $y = g^x \pmod p$.
2. **Signing:** For message $m$ with hash $h = \text{SHA-256}(m) \pmod{(p-1)}$:
   - Choose ephemeral nonce $k \in [2, p-2]$ such that $\gcd(k, p-1) = 1$.
   - Compute $r = g^k \pmod p$.
   - Compute $s = k^{-1}(h - x \cdot r) \pmod{p-1}$. The signature is $(r, s)$.
3. **Verification:** Valid iff $0 < r < p$, $0 < s < p-1$, and:
$$g^h \equiv y^r \cdot r^s \pmod p$$

## 4.4. Catastrophic Ephemeral Nonce k-reuse Attack
If a signer reuses the same ephemeral nonce $k$ for two distinct messages $m_1, m_2$ with hashes $h_1, h_2$:
$$s_1 \equiv k^{-1}(h_1 - x \cdot r) \pmod{p-1}, \quad s_2 \equiv k^{-1}(h_2 - x \cdot r) \pmod{p-1}$$
Subtracting the two equations eliminates the private key $x$:
$$(s_1 - s_2) \cdot k \equiv (h_1 - h_2) \pmod{p-1}$$
Solving this congruence recovers $k$. Once $k$ is known, the private key $x$ is immediately recovered:
$$x \equiv (h_1 - k \cdot s_1) \cdot r^{-1} \pmod{p-1}$$


In [13]:
# ElGamal Verification: Domain Parameters, Keypair, Signing, Verification, and k-reuse Attack
import secrets
from src.elgamal import generate_elgamal_params, generate_elgamal_keypair, elgamal_sign, elgamal_verify
from src.sha256 import sha256_bytes
from src.rsa import extended_gcd, mod_inverse

# 1. Generate 128-bit safe prime parameters & keypair
p, g = generate_elgamal_params(bits=128)
x, y = generate_elgamal_keypair(p, g)
print(f"[+] Safe Prime p ({p.bit_length()} bits): {p}")
print(f"[+] Generator g: {g}")
print(f"[+] Public Key y: {y}")

# 2. Sign and Verify
msg = b"Electronic Transaction Contract - Student: Nguyen Thi Nhu Y"
r, s = elgamal_sign(msg, x, p, g)
valid = elgamal_verify(msg, (r, s), y, p, g)
print(f"[+] Signature Generation: r = {r}, s = {s}")
print(f"[+] Signature Verification Result: {valid} (Expected True)")

# 3. Tamper tests
tampered_msg = b"Electronic Transaction Contract - Student: Nguyen Thi Nhu Z"
print(f"[+] Tampered Message Rejected: {elgamal_verify(tampered_msg, (r, s), y, p, g) is False}")
print(f"[+] Tampered Signature r Rejected: {elgamal_verify(msg, (r ^ 1, s), y, p, g) is False}")

# 4. Ephemeral nonce k-reuse attack demonstration
m1 = b"Transaction A: 10,000,000 VND"
m2 = b"Transaction B: 50,000,000 VND"
h1 = int.from_bytes(sha256_bytes(m1), 'big')
h2 = int.from_bytes(sha256_bytes(m2), 'big')
p_minus_1 = p - 1

k_fixed = secrets.randbelow(p - 3) + 2
while extended_gcd(k_fixed, p_minus_1)[0] != 1:
    k_fixed = secrets.randbelow(p - 3) + 2

r_fixed = pow(g, k_fixed, p)
k_inv = mod_inverse(k_fixed, p_minus_1)
s1 = (k_inv * (h1 - x * r_fixed)) % p_minus_1
s2 = (k_inv * (h2 - x * r_fixed)) % p_minus_1

delta_s = (s1 - s2) % p_minus_1
delta_h = (h1 - h2) % p_minus_1
g_gcd, _, _ = extended_gcd(delta_s, p_minus_1)
if g_gcd == 1:
    k_rec = (delta_h * mod_inverse(delta_s, p_minus_1)) % p_minus_1
else:
    mod_red = p_minus_1 // g_gcd
    k_base = ((delta_h // g_gcd) * mod_inverse(delta_s // g_gcd, mod_red)) % mod_red
    k_rec = next(cand for step in range(g_gcd) if pow(g, (cand := k_base + step * mod_red), p) == r_fixed)

rhs = (h1 - k_rec * s1) % p_minus_1
g_r, _, _ = extended_gcd(r_fixed, p_minus_1)
if g_r == 1:
    x_rec = (rhs * mod_inverse(r_fixed, p_minus_1)) % p_minus_1
else:
    mod_r_red = p_minus_1 // g_r
    x_base = ((rhs // g_r) * mod_inverse(r_fixed // g_r, mod_r_red)) % mod_r_red
    x_rec = next(cand_x for step in range(g_r) if pow(g, (cand_x := x_base + step * mod_r_red), p) == y)

print(f"[!] Ephemeral Key k Compromised: {k_rec == k_fixed} (Recovered k: {k_rec})")
print(f"[!] Private Key x Compromised: {x_rec == x} (Recovered x: {x_rec})")


[+] Safe Prime p (128 bits): 324109181089483556749448603000598229763
[+] Generator g: 35663281104375280063313911443886477014
[+] Public Key y: 242426225380287779821229134668873002116
[+] Signature Generation: r = 59746404480492242171871772674933956153, s = 107198994663383585377738618877194394842
[+] Signature Verification Result: True (Expected True)
[+] Tampered Message Rejected: True
[+] Tampered Signature r Rejected: True
[!] Ephemeral Key k Compromised: True (Recovered k: 74141971108584430139877792902747104643)
[!] Private Key x Compromised: True (Recovered x: 9509307861289641505797386185935248210)


## 4.8. Empirical Evaluation and Experimental Test Results: ElGamal
- **Automated Unit Testing:** 12 out of 12 tests in `tests/test_elgamal.py` PASSED completely.
- **Benchmark Performance:**
  - **512-bit Domain:** Safe prime generation: 881.67 ms (0.96 ops/s); Keygen: 0.46 ms (2,161 ops/s); Sign: 0.82 ms (1,162 ops/s); Verify: 1.29 ms (748 ops/s).
  - **1024-bit Domain:** Safe prime generation: 21,248.60 ms (0.04 ops/s); Keygen: 2.66 ms (373 ops/s); Sign: 3.37 ms (287 ops/s); Verify: 6.23 ms (158 ops/s).
  - **Signature Size:** 128 bytes for 512-bit domain, 256 bytes for 1024-bit domain (representing $(r, s)$).

### Empirical Terminal Screenshot: ElGamal Benchmark Execution
![ElGamal Benchmark Terminal](evidence/benchmark_elgamal.png)

### Empirical Terminal Screenshot: ElGamal Interactive Demo Execution
![ElGamal Demo Terminal](evidence/demo_elgamal.jpg)


---
# SECTION 5: COMPREHENSIVE PERFORMANCE BENCHMARK & TEST EXECUTION

All empirical benchmark measurements were gathered systematically by `benchmark.py` using `time.perf_counter()` on Windows 11 with Python 3.13.15 across 28 distinct operations.
The full benchmark suite completed in **324.716 seconds** wall time.


In [16]:
# Load and format benchmark results from benchmark_results.csv
import csv

print(f"{'Algorithm':<10} | {'Operation':<16} | {'Size/Key':<10} | {'n':<4} | {'Median (ms)':<12} | {'Throughput'}")
print("-" * 78)
with open('benchmark_results.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        ops = float(row['ops_per_s'])
        algo = row['algorithm']
        op = row['operation']
        size_str = row['size']
        mean_ms = float(row['mean_ms'])
        
        if algo in ('AES-128', 'SHA-256') and any(k in op for k in ('cbc', 'hash', 'encrypt', 'decrypt')):
            if 'KB' in size_str:
                kb = float(size_str.replace(' KB', ''))
                mb_s = (kb / 1024.0) / (mean_ms / 1000.0)
                throughput_str = f"{mb_s:.3f} MB/s"
            elif 'MB' in size_str:
                mb = float(size_str.replace(' MB', ''))
                mb_s = mb / (mean_ms / 1000.0)
                throughput_str = f"{mb_s:.3f} MB/s"
            else:
                throughput_str = f"{ops:.1f} ops/s"
        else:
            throughput_str = f"{ops:.1f} ops/s"
        print(f"{algo:<10} | {op:<16} | {size_str:<10} | {row['n']:<4} | {float(row['median_ms']):<12.4f} | {throughput_str}")


Algorithm  | Operation        | Size/Key   | n    | Median (ms)  | Throughput
------------------------------------------------------------------------------
AES-128    | key_expansion    | 16 bytes   | 50   | 0.0263       | 37207.9 ops/s
AES-128    | encrypt_cbc      | 1 KB       | 10   | 3.4492       | 0.274 MB/s
AES-128    | decrypt_cbc      | 1 KB       | 10   | 35.6849      | 0.026 MB/s
AES-128    | encrypt_cbc      | 16 KB      | 10   | 55.4176      | 0.279 MB/s
AES-128    | decrypt_cbc      | 16 KB      | 10   | 584.4326     | 0.027 MB/s
AES-128    | encrypt_cbc      | 64 KB      | 10   | 226.0818     | 0.278 MB/s
AES-128    | decrypt_cbc      | 64 KB      | 10   | 2328.4935    | 0.027 MB/s
RSA        | keygen           | 512 bits   | 30   | 18.2338      | 53.3 ops/s
RSA        | encrypt          | 512 bits   | 30   | 0.0121       | 81124.9 ops/s
RSA        | decrypt          | 512 bits   | 30   | 0.4673       | 2046.6 ops/s
RSA        | keygen           | 1024 bits  | 30   | 166

## 5.2. In-Notebook Automated Unit Test Suite Execution
The code block below dynamically discovers and executes all 65 automated unit tests across the 4 cryptographic algorithms using Python's `unittest` framework.


In [18]:
# Discover and run all 65 unit test cases
import os
import sys
import unittest

sys.path.insert(0, os.path.abspath('.'))
loader = unittest.TestLoader()
suite = loader.discover('tests')
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print(f"\n{'='*70}")
print(f"UNIT TEST SUMMARY: Ran {result.testsRun} tests | Failures: {len(result.failures)} | Errors: {len(result.errors)}")
print(f"Overall Status: {'PASSED (100% OK)' if result.wasSuccessful() else 'FAILED'}")
print(f"{'='*70}")



UNIT TEST SUMMARY: Ran 65 tests | Failures: 0 | Errors: 0
Overall Status: PASSED (100% OK)


test_add_round_key_fips197_appendix_b (test_aes128.TestAES128.test_add_round_key_fips197_appendix_b)
Test AddRoundKey with round key 1 against FIPS 197 Appendix B Round 1. ... ok
test_add_round_key_twice_returns_original (test_aes128.TestAES128.test_add_round_key_twice_returns_original)
Edge test: Applying AddRoundKey twice with the same key returns original state. ... ok
test_aes_transformations_roundtrip (test_aes128.TestAES128.test_aes_transformations_roundtrip)
Test roundtrips for SubBytes, ShiftRows, and MixColumns. ... ok
test_cbc_pkcs7_regression_vectors (test_aes128.TestAES128.test_cbc_pkcs7_regression_vectors)
Test CBC encryption with PKCS#7 against regression vectors with student key. ... ok
test_cbc_pkcs7_roundtrips_variable_lengths (test_aes128.TestAES128.test_cbc_pkcs7_roundtrips_variable_lengths)
Test roundtrip for lengths 0, 1, 15, 16, 17, 31, 32, 100 bytes. ... ok
test_decrypt_block_fips197_appendix_b (test_aes128.TestAES128.test_decrypt_block_fips197_appendix_b)
Test d

---
# SECTION 6: CONCLUSION AND ACADEMIC INTEGRITY

## 6.1. Project Accomplishments
The project has thoroughly researched and implemented four foundational cryptographic algorithms from first principles in pure Python 3.13:
1. **AES-128:** Full round transformations on $\text{GF}(2^8)$, key expansion, and CBC block cipher mode with PKCS#7 padding.
2. **RSA:** Custom Miller-Rabin primality testing ($k=40$), Extended Euclidean algorithm, modular inversion, and textbook public-key encryption/decryption.
3. **SHA-256:** NIST FIPS 180-4 compliant 64-word message schedule, 64-round compression loop, and empirical verification of the Strict Avalanche Criterion (~52.34%).
4. **ElGamal Digital Signatures:** Safe prime parameter generation, primitive root generators, randomized signing, tamper rejection, and mathematical demonstration of the catastrophic $k$-reuse attack.
All 65 automated unit test cases pass with a 100% success rate.

## 6.2. Implementation Limitations
- Developed strictly for educational and pedagogical purposes to illuminate mathematical and algorithmic principles.
- Lacks constant-time execution, leaving it susceptible to timing side-channel attacks.
- Textbook RSA lacks OAEP/PSS randomized padding schemes required in production.
- Pure Python implementation throughput is lower than optimized C/assembly libraries (e.g., OpenSSL).

## 6.3. Academic Integrity Commitment
The student certifies that all source code and documentation were developed independently from scratch without third-party cryptographic libraries, maintaining strict academic honesty.

---
# SECTION 7: REFERENCES

1. NIST FIPS PUB 197 (2001), *Advanced Encryption Standard (AES)*, National Institute of Standards and Technology.
2. NIST SP 800-38A (2001), *Recommendation for Block Cipher Modes of Operation*, National Institute of Standards and Technology.
3. RFC 8017 (2016), *PKCS #1: RSA Cryptography Specifications Version 2.2*, Internet Engineering Task Force.
4. NIST FIPS PUB 180-4 (2015), *Secure Hash Standard (SHS)*, National Institute of Standards and Technology.
5. Taher Elgamal (1985), *A Public Key Cryptosystem and a Signature Scheme Based on Discrete Logarithms*, IEEE Transactions on Information Theory.
6. Ronald Rivest, Adi Shamir, Leonard Adleman (1978), *A Method for Obtaining Digital Signatures and Public-Key Cryptosystems*, Communications of the ACM.
7. Michael O. Rabin (1980), *Probabilistic Algorithm for Testing Primality*, Journal of Number Theory.
8. William Stallings (2017), *Cryptography and Network Security: Principles and Practice (7th Edition)*, Pearson.
